## 通过 Pytorch 来手工实现 Scaled Dot-product Attention

### 构建一个从 token ID 到 token embedding 的映射表

In [11]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from torch import nn
from transformers import AutoConfig
from transformers import AutoTokenizer
import tensorflow as tf
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"] = "0"    #这里是gpu的序号，指定使用的gpu对象
config = tf.ConfigProto()
config.gpu_options.allow_growth = True


ModuleNotFoundError: No module named 'torch'

In [9]:
text = "time flies like an arrow"
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
print(inputs.input_ids)

tensor([[ 2051, 10029,  2066,  2019,  8612]])


In [10]:
config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
print(token_emb)

Embedding(30522, 768)


In [11]:
inputs_embeds = token_emb(inputs.input_ids)
print(inputs_embeds.size())
inputs_embeds

torch.Size([1, 5, 768])


tensor([[[-1.9035,  0.9345, -1.7802,  ..., -0.9183, -0.7262,  0.4975],
         [-1.0038, -0.7815, -0.8151,  ..., -0.6116, -0.6024,  0.1289],
         [-0.0616,  0.7412,  0.1297,  ..., -0.0662, -2.2118,  0.3655],
         [-2.0734, -1.8088,  1.9643,  ..., -0.3186,  0.9854, -0.9318],
         [ 0.1453,  0.7818,  0.8754,  ...,  1.0678,  0.9309, -0.9171]]],
       grad_fn=<EmbeddingBackward0>)

BERT-base-uncased 模型对应的词表大小为 30522，每个词语的词向量维度为 768

### 接下来就是创建 query、key、value 向量序列，并且使用点积作为相似度函数来计算注意力分数：

In [12]:
import torch
from math import sqrt

Q = K = V = inputs_embeds
dim_k = K.size(-1)
scores = torch.bmm(Q, K.transpose(1,2)) / sqrt(dim_k)
dim_k
scores
torch.Size

768

tensor([[[28.2740,  0.7452, -0.6104, -1.6921, -0.7816],
         [ 0.7452, 29.8184,  2.3867, -0.3844, -2.6730],
         [-0.6104,  2.3867, 29.6866,  0.9311, -0.1095],
         [-1.6921, -0.3844,  0.9311, 28.0674, -1.4641],
         [-0.7816, -2.6730, -0.1095, -1.4641, 28.2534]]],
       grad_fn=<DivBackward0>)

torch.Size

In [14]:
import torch.nn.functional as F

weights = F.softmax(scores, dim=-1)
print(weights.sum(dim=-1))

tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)


In [16]:
attn_outputs = torch.bmm(weights, V)
print(attn_outputs.shape)
attn_outputs

torch.Size([1, 5, 768])


tensor([[[-1.9035,  0.9345, -1.7802,  ..., -0.9183, -0.7262,  0.4975],
         [-1.0038, -0.7815, -0.8151,  ..., -0.6116, -0.6024,  0.1289],
         [-0.0616,  0.7412,  0.1297,  ..., -0.0662, -2.2118,  0.3655],
         [-2.0734, -1.8088,  1.9643,  ..., -0.3186,  0.9854, -0.9318],
         [ 0.1453,  0.7818,  0.8754,  ...,  1.0678,  0.9309, -0.9171]]],
       grad_fn=<BmmBackward0>)

### 将上面这些操作封装为函数以方便后续调用

In [17]:
import torch
import torch.nn.functional as F
from math import sqrt

def scaled_dot_product_attention(query, key, value, query_mask=None, key_mask=None, mask=None):
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    if query_mask is not None and key_mask is not None:
        mask = torch.bmm(query_mask.unsqueeze(-1), key_mask.unsqueeze(1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -float("inf"))
    weights = F.softmax(scores, dim=-1)
    return torch.bmm(weights, value)

### Multi-head Attention

In [18]:
from torch import nn

class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.q = nn.Linear(embed_dim, head_dim)
        self.k = nn.Linear(embed_dim, head_dim)
        self.v = nn.Linear(embed_dim, head_dim)

    def forward(self, query, key, value, query_mask=None, key_mask=None, mask=None):
        attn_outputs = scaled_dot_product_attention(
            self.q(query), self.k(key), self.v(value), query_mask, key_mask, mask)
        return attn_outputs

### 实践中一般将 head_dim 设置为 embed_dim 的因数，这样 token 嵌入式表示的维度就可以保持不变，例如 BERT 有 12 个注意力头，因此每个头的维度被设置为 768/12。

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.hidden_size
        num_heads = config.num_attention_heads
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
        )
        self.output_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, query_mask=None, key_mask=None, mask=None):
        x = torch.cat([
            h(query, key, value, query_mask, key_mask, mask) for h in self.heads
        ], dim=-1)
        x = self.output_linear(x)
        return x

In [21]:
from transformers import AutoConfig
from transformers import AutoTokenizer

model_ckpt = "D:/DATA/jupyter/ZHY/transformer/Tersor_model/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

text = "time flies like an arrow"
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
config = AutoConfig.from_pretrained(model_ckpt)

token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
inputs_embeds = token_emb(inputs.input_ids)

multihead_attn = MultiHeadAttention(config)
query = key = value = inputs_embeds
attn_output = multihead_attn(query, key, value)
print(attn_output.size())

torch.Size([1, 5, 768])


# The Feed-Forward Layer

In [22]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.linear_2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x
feed_forward = FeedForward(config)
ff_outputs = feed_forward(attn_output)
print(ff_outputs.size())

torch.Size([1, 5, 768])


# eat coding

In [34]:
import math
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as Data
#device = 'cpu'
device = 'cuda'

# transformer epochs
epochs = 100
# epochs = 1000

# 这里我没有用什么大型的数据集，而是手动输入了两对中文→英语的句子
# 还有每个字的索引也是我手动硬编码上去的，主要是为了降低代码阅读难度
# S: Symbol that shows starting of decoding input
# E: Symbol that shows starting of decoding output
# P: Symbol that will fill in blank sequence if current batch data size is shorter than time steps

In [35]:
# 训练集
sentences = [
    # 中文和英语的单词个数不要求相同
    # enc_input                dec_input           dec_output
    ['我 有 一 个 女 朋 友', 'S i have a girl friend . ', 'i have a girl friend . E'],
    ['我 有 零 个 好 朋 友', 'S i have zero good friend .', 'i have zero good friend . E']
]
# 中文和英语的单词要分开建立词库
# Padding Should be Zero
src_vocab = {
            'P': 0, '我': 1, '有': 2, '一': 3, '个': 4, '好': 5, '朋': 6, '友': 7, '零': 8, '女': 9}
src_idx2word = {
            i: w for i, w in enumerate(src_vocab)}
src_vocab_size = len(src_vocab)

tgt_vocab = {
            'P': 0, 'i': 1, 'have': 2, 'a': 3, 'good': 4, 'friend': 5, 'zero': 6, 'girl': 7, 'S': 8, 'E': 9, '.': 10}
idx2word = {
            i: w for i, w in enumerate(tgt_vocab)}

tgt_vocab_size = len(tgt_vocab)

src_len = 8  # （源句子的长度）enc_input max sequence length
tgt_len = 7  # dec_input(=dec_output) max sequence length
# Transformer Parameters
d_model = 512  # Embedding Size（token embedding和position编码的维度）
d_ff = 2048  # FeedForward dimension (两次线性层中的隐藏层 512->2048->512，线性层是用来做特征提取的），当然最后会再接一个projection层
d_k = d_v = 64  # dimension of K(=Q), V（Q和K的维度需要相同，这里为了方便让K=V）
n_layers = 6  # number of Encoder of Decoder Layer（Block的个数）
n_heads = 8  # number of heads in Multi-Head Attention（有几套头）


In [36]:
# 数据构建


def make_data(sentences):
    """把单词序列转换为数字序列"""
    enc_inputs, dec_inputs, dec_outputs = [], [], []
    for i in range(len(sentences)):
        enc_input = [[src_vocab[n] for n in sentences[i][0].split()]]  # [[1, 2, 3, 4, 0], [1, 2, 3, 5, 0]]
        dec_input = [[tgt_vocab[n] for n in sentences[i][1].split()]]  # [[6, 1, 2, 3, 4, 8], [6, 1, 2, 3, 5, 8]]
        dec_output = [[tgt_vocab[n] for n in sentences[i][2].split()]]  # [[1, 2, 3, 4, 8, 7], [1, 2, 3, 5, 8, 7]]

        enc_inputs.extend(enc_input)
        dec_inputs.extend(dec_input)
        dec_outputs.extend(dec_output)

    return torch.LongTensor(enc_inputs), torch.LongTensor(dec_inputs), torch.LongTensor(dec_outputs)

enc_inputs, dec_inputs, dec_outputs = make_data(sentences)
enc_inputs
dec_inputs
dec_outputs


tensor([[1, 2, 3, 4, 9, 6, 7],
        [1, 2, 8, 4, 5, 6, 7]])

tensor([[ 8,  1,  2,  3,  7,  5, 10],
        [ 8,  1,  2,  6,  4,  5, 10]])

tensor([[ 1,  2,  3,  7,  5, 10,  9],
        [ 1,  2,  6,  4,  5, 10,  9]])

In [37]:
class MyDataSet(Data.Dataset):
    """自定义DataLoader"""

    def __init__(self, enc_inputs, dec_inputs, dec_outputs):
        super(MyDataSet, self).__init__()
        self.enc_inputs = enc_inputs
        self.dec_inputs = dec_inputs
        self.dec_outputs = dec_outputs

    def __len__(self):
        return self.enc_inputs.shape[0]

    def __getitem__(self, idx):
        return self.enc_inputs[idx], self.dec_inputs[idx], self.dec_outputs[idx]

loader = Data.DataLoader(MyDataSet(enc_inputs, dec_inputs, dec_outputs), 2, True)
# 这行代码创建了一个PyTorch数据加载器，用于在训练机器学习模型时加载数据。DataLoader是PyTorch的核心库torch.utils.data中的函数。它的作用是将
# 数据集（此处为MyDataSet类的实例，传入的enc_inputs、dec_inputs和dec_outputs作为数据）拆分成小批次以提高加载效率。
# 具体参数说明：
# batch_size: 2，每批加载的样本数。
# shuffle: True，是否打乱数据顺序。

模型构建
Positional Encoding

In [4]:
device='cuda:0'

In [40]:
import tensorflow as tf

# 获取 TensorFlow 的设备列表
devices = tf.config.experimental.list_physical_devices('GPU')

if len(devices) > 0:
    # 检测到 GPU 设备
    print("代码正在GPU上运行")
else:
    print("代码正在CPU上运行")

ModuleNotFoundError: No module named 'tensorflow'

In [7]:
from tensorflow.python.client import device_lib
 
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 17525518255429417381
xla_global_id: -1
]


In [8]:
device = 'cuda'

In [9]:
from tensorflow.python.client import device_lib
 
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 12882157523565766436
xla_global_id: -1
]


In [10]:
import tensorflow as tf
tf.config.experimental.list_physical_devices('GPU')

[]

In [9]:
#导入相关的库
#import tensorflow.contrib.slim as slim 注释掉原代码，改从tf_slim 中调用slim
import tf_slim as slim


import tensorflow as tf
from tensorflow.examples.tutorials.mnist import input_data
import os
import time
from tensorflow.contrib.tensorboard.plugins import projector
import matplotlib.pyplot as plt
import numpy as np
#这里用slim这个API来进行卷积网络构建
slim = tf.contrib.slim

#定义卷积神经网络模型
#网络架构是卷积网络--最大池化--卷积网络--最大池化---flatten---MLP-softmax的全连接MLP
def model(inputs, is_training, dropout_rate, num_classes, scope='Net'):
    config=tf.ConfigProto(allow_soft_placement=True)#修改这里
    inputs = tf.reshape(inputs, [-1, 28, 28, 1])
    with tf.variable_scope(scope):
        with slim.arg_scope([slim.conv2d, slim.fully_connected],
                            normalizer_fn=slim.batch_norm):
            net = slim.conv2d(inputs, 32, [5, 5], padding='SAME', scope='conv1')
            net = slim.max_pool2d(net, 2, stride=2, scope='maxpool1')
            tf.summary.histogram("conv1", net)

            net = slim.conv2d(net, 64, [5, 5], padding='SAME', scope='conv2')
            net = slim.max_pool2d(net, 2, stride=2, scope='maxpool2')
            tf.summary.histogram("conv2", net)

            net = slim.flatten(net, scope='flatten')
            fc1 = slim.fully_connected(net, 1024, scope='fc1')
            tf.summary.histogram("fc1", fc1)

            net = slim.dropout(fc1, dropout_rate, is_training=is_training, scope='fc1-dropout')
            net = slim.fully_connected(net, num_classes, scope='fc2')

            return net, fc1


def create_sprite_image(images):
    """更改图片的shape"""
    if isinstance(images, list):
        images = np.array(images)
    img_h = images.shape[1]
    img_w = images.shape[2]
    n_plots = int(np.ceil(np.sqrt(images.shape[0])))

    sprite_image = np.ones((img_h * n_plots, img_w * n_plots))

    for i in range(n_plots):
        for j in range(n_plots):
            this_filter = i * n_plots + j
            if this_filter < images.shape[0]:
                this_img = images[this_filter]
                sprite_image[i * img_h:(i + 1) * img_h,
                j * img_w:(j + 1) * img_w] = this_img

    return sprite_image


def vector_to_matrix_mnist(mnist_digits):
    """把正常的mnist数字图片(batch,28*28)这个格式，转换为新的张量形状(batch,28,28)"""
    return np.reshape(mnist_digits, (-1, 28, 28))


def invert_grayscale(mnist_digits):
    """处理下图片颜色，黑色变白，白色边黑"""
    return 1 - mnist_digits


if __name__ == "__main__":
    # 定义参数
    #学习率
    learning_rate = 1e-4
    #定义迭代参数
    total_epoch = 600
    #定义批量
    batch_size = 200
    #程序运行中打印频率
    display_step = 20
    #程序运行中保存结果的频率
    save_step = 100
    load_checkpoint = False
    checkpoint_dir = "checkpoint"
    checkpoint_name = 'model.ckpt'
    #结果存放的路径
    logs_path = "logs"
    #定义我们使用多少个图片
    test_size = 10000
    #定义第二层路径
    projector_path = 'projector'

    # 网络参数
    n_input = 28 * 28   # 每个图片是28*28个像素，也就是784个特征
    n_classes = 10  # MNIST数据集有0-9是个类别的结果
    dropout_rate = 0.5  # Dropout的比率

    mnist = input_data.read_data_sets('MNIST-data', one_hot=True)

    # 定义计算图
    x = tf.placeholder(tf.float32, [None, n_input], name='InputData')
    y = tf.placeholder(tf.float32, [None, n_classes], name='LabelData')
    is_training = tf.placeholder(tf.bool, name='IsTraining')
    keep_prob = dropout_rate

    logits, fc1 = model(x, is_training, keep_prob, n_classes)

    with tf.name_scope('Loss'):
        loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=y))
    tf.summary.scalar("loss", loss)

    with tf.name_scope('Accuracy'):
        correct_pred = tf.equal(tf.argmax(logits, 1), tf.argmax(y, 1))
        accuracy = tf.reduce_mean(tf.cast(correct_pred, tf.float32))
    tf.summary.scalar("accuracy", accuracy)

    optimizer = tf.train.AdamOptimizer(learning_rate=learning_rate).minimize(loss)

    projector_dir = os.path.join(logs_path, projector_path)
    path_metadata = os.path.join(projector_dir,'metadata.tsv')
    path_sprites = os.path.join(projector_dir, 'mnistdigits.png')
    # 检查结果目录的状态
    if not os.path.exists(projector_dir):
        os.makedirs(projector_dir)

    # 这里进行嵌入
    mnist_test = input_data.read_data_sets('MNIST-data', one_hot=False)
    batch_x_test = mnist_test.test.images[:test_size]
    batch_y_test = mnist_test.test.labels[:test_size]

    embedding_var = tf.Variable(tf.zeros([test_size, 1024]), name='embedding')
    assignment = embedding_var.assign(fc1)

    config = projector.ProjectorConfig()
    embedding = config.embeddings.add()
    embedding.tensor_name = embedding_var.name
    embedding.metadata_path = os.path.join(projector_path,'metadata.tsv')
    embedding.sprite.image_path = os.path.join(projector_path, 'mnistdigits.png')
    embedding.sprite.single_image_dim.extend([28,28])

    # 初始化变量
    init = tf.global_variables_initializer()

    # 'Saver' op to save and restore all the variables
    saver = tf.train.Saver()
    merged_summary_op = tf.summary.merge_all()

    # 运行计算图
    with tf.Session() as sess:
        sess.run(init)
        # Restore model weights from previously saved model
        prev_model = tf.train.get_checkpoint_state(checkpoint_dir)
        if load_checkpoint:
            if prev_model:
                saver.restore(sess, prev_model.model_checkpoint_path)
                print('Checkpoint found, {}'.format(prev_model))
            else:
                print('No checkpoint found')

        summary_writer = tf.summary.FileWriter(logs_path, graph=tf.get_default_graph())
        projector.visualize_embeddings(summary_writer, config)
        start_time = time.time()
        # 开始训练
        for epoch in range(total_epoch):
            batch_x, batch_y = mnist.train.next_batch(batch_size)
            # reshapeX = np.reshape(batch_x, [-1, 28, 28, 1])
            # 开始反向传播算法
            sess.run(optimizer, feed_dict={x: batch_x, y: batch_y,
                                           is_training: True})
            if epoch % display_step == 0:
                # 计算损失和精度
                cost, acc, summary = sess.run([loss, accuracy, merged_summary_op],
                                              feed_dict={x: batch_x,
                                                         y: batch_y,
                                                         is_training: False})
                elapsed_time = time.time() - start_time
                start_time = time.time()
                print('epoch {}, training accuracy: {:.4f}, loss: {:.5f}, time: {}'
                      .format(epoch, acc, cost, elapsed_time))
                summary_writer.add_summary(summary, epoch)
            if epoch % save_step == 0:
                # 保存训练的结果
                sess.run(assignment, feed_dict={x: mnist.test.images[:test_size],
                                                y: mnist.test.labels[:test_size], is_training: False})
                checkpoint_path = os.path.join(checkpoint_dir, checkpoint_name)
                save_path = saver.save(sess, checkpoint_path)
                print("Model saved in file: {}".format(save_path))

        # 保存结果
        saver.save(sess, os.path.join(logs_path, "model.ckpt"), 1)
        # 创建可视化的图片
        to_visualise = batch_x_test
        to_visualise = vector_to_matrix_mnist(to_visualise)
        to_visualise = invert_grayscale(to_visualise)
        sprite_image = create_sprite_image(to_visualise)
        # 保存可视化的图片
        plt.imsave(path_sprites, sprite_image, cmap='gray')
        # 写文件
        with open(path_metadata, 'w') as f:
            f.write("Index\tLabel\n")
            for index, label in enumerate(batch_y_test):
                f.write("%d\t%d\n" % (index, label))

        print("训练完成")

Extracting MNIST-data\train-images-idx3-ubyte.gz
Extracting MNIST-data\train-labels-idx1-ubyte.gz
Extracting MNIST-data\t10k-images-idx3-ubyte.gz
Extracting MNIST-data\t10k-labels-idx1-ubyte.gz


ValueError: Variable Net/conv1/weights already exists, disallowed. Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope? Originally defined at:

  File "D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorflow\python\framework\ops.py", line 2005, in __init__
    self._traceback = tf_stack.extract_stack()
  File "D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorflow\python\framework\ops.py", line 3616, in create_op
    op_def=op_def)
  File "D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorflow\python\util\deprecation.py", line 507, in new_func
    return func(*args, **kwargs)
  File "D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorflow\python\framework\op_def_library.py", line 788, in _apply_op_helper
    op_def=op_def)
  File "D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorflow\python\ops\gen_state_ops.py", line 2024, in variable_v2
    shared_name=shared_name, name=name)


In [4]:
import tensorflow as tf

print(tf.version)

<module 'tensorflow._api.v1.version' from 'D:\\Software\\Anaconda3\\envs\\tensorflow1.14\\lib\\site-packages\\tensorflow\\_api\\v1\\version\\__init__.py'>


D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorboard\compat\tensorflow_stub\dtypes.py:541: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorboard\compat\tensorflow_stub\dtypes.py:542: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
D:\Software\Anaconda3\envs\tensorflow1.14\lib\site-packages\tensorboard\compat\tensorflow_stub\dtypes.py:543: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
D:\Software\Anaconda3\envs\tensorf

In [10]:
import tensorflow as tf
import numpy as np
import os
os.environ["CUDA_DEVICES_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
 
with tf.device("/gpu:0"):
# with tf.device("/job:localhost/replica:0/task:0/device:GPU:0"):
    a = tf.Variable(0,dtype=tf.float32)
 
with tf.device("/gpu:1"):
    b = tf.constant(0, shape=[1], dtype=tf.float32)
 
 
#config = tf.ConfigProto()
config = tf.ConfigProto(allow_soft_placement=True)
config.gpu_options.allow_growth = True
config.log_device_placement = True
with tf.Session(config=config) as sess:
    sess.run(tf.global_variables_initializer())
    c = sess.run(a)
    b_ = sess.run(b)
    print(b_,c)

[0.] 0.0


In [13]:
import tensorflow as tf
import os
# 降低警告优先级，去掉一堆加速警告。可以不写
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

gpu_device_name = tf.test.gpu_device_name()
print(gpu_device_name)

print(tf.test.is_gpu_available())  # 返回True或者False


/device:GPU:0
True
